<h1 style=\"text-align: center; font-size: 50px;\">😷 Register Model for COVID Movement Patterns with VAR (Vector Autoregression)</h1>
This notebook shows an visual data analysis of the effects of COVID-19 in two different cities: New York and London

## Notebook Overview
- Imports
- Configurations
- Logging Model to MLflow
- Fetching the Latest Model Version from MLflow
- Loading the Model and Running Inference

## Imports

In [1]:
%%time

%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.
CPU times: user 16.6 ms, sys: 1.53 ms, total: 18.1 ms
Wall time: 631 ms


In [2]:
# ------------------------ Data Manipulation ------------------------
import pandas as pd
import numpy as np

# ------------------------ System Utilities ------------------------
import warnings
import logging
from pathlib import Path
import os
import pickle
import time
import json
from typing import Any, Optional, Dict
import sys

# ------------------------ MLflow for Experiment Tracking and Model Management ------------------------
import mlflow
from mlflow import MlflowClient
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec, TensorSpec, ParamSchema, ParamSpec

# ------------------------ Statistical Analysis ------------------------
from statsmodels.tsa.stattools import adfuller

# ------------------------ New Models-from-Code Integration ------------------------
# Define the relative path to the 'src' directory (two levels up from current working directory)
src_path = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Add 'src' directory to system path for module imports (e.g., utils)
if src_path not in sys.path:
    sys.path.append(src_path)

from src.utils import (
    load_config,)
from src.mlflow import Logger

## Configurations

In [3]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [4]:
# Create logger
logger = logging.getLogger("cities_analysis_logger")
logger.setLevel(logging.INFO)
logger.propagate = False
logger.handlers.clear()

formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s", 
                              datefmt="%Y-%m-%d %H:%M:%S")  

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [5]:
# ------------------------- Paths -------------------------
DATA_PATH = "/home/jovyan/datafabric/tutorial/"
ARTIFACTS_PATH = "../artifacts"
DEMO_FOLDER = "../demo"
CONFIG_PATH = "../configs/config.yaml"

# ------------------------ MLflow Integration ------------------------
EXPERIMENT_NAME = "Two_Cities_Experiment"
RUN_NAME = "Two_Cities_Run"
MODEL_NAME = "Two_Cities_Model"

In [6]:
start_time = time.time()  
logger.info('Notebook execution started.')

2025-10-14 19:37:20 - INFO - Notebook execution started.


## Verify Assets


In [7]:
# Check whether the Dataset file exists
is_dataset_available = Path(DATA_PATH).exists()

# Log the configuration status of the dataset
if is_dataset_available:
    logger.info("The Dataset is properly configured.")
else:
    logger.info(
        "The Dataset is not properly configured. Please create and download the required assets "
        "in your project on AI Studio."
    )

config = load_config(CONFIG_PATH)

print("✅ Configuration loaded successfully")


2025-10-14 19:37:20 - INFO - The Dataset is properly configured.


✅ Configuration loaded successfully


## Logging Model to MLflow

Reading the JSON with training parameters that were saved during training

In [8]:
artifacts_dir = ARTIFACTS_PATH
os.makedirs(artifacts_dir, exist_ok=True)

with open(f"{artifacts_dir}/training_metrics.json", "r") as metrics:
            metrics_dict = json.load(metrics)

In [9]:
# Define input and output schema for model signature
input_schema = Schema([
    ColSpec("string","city"),
    ColSpec("long","steps"),
    ])
output_schema = Schema([
    ColSpec("string", "class"),
])

# Define model signature
signature = ModelSignature(inputs=input_schema, outputs=output_schema)

In [10]:
logger.info(f'Starting the experiment: {EXPERIMENT_NAME}')

mlflow.set_tracking_uri("/phoenix/mlflow")
# Set the MLflow experiment name
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

# Start an MLflow run
with mlflow.start_run(run_name=RUN_NAME) as run:    
    
    # Registering the training metrics to mlflow
    mlflow.log_metrics(metrics_dict)
    
    # Print the artifact URI for reference
    logger.info(f"Run's Artifact URI: {run.info.artifact_uri}")
    
    # Log the model using new models-from-code approach
    
    Logger.log_model(
        signature=signature,
        artifact_path=MODEL_NAME,
        config_path=CONFIG_PATH,
        docs_path=ARTIFACTS_PATH,  # This contains the pickle files
        demo_folder=DEMO_FOLDER
    )

    # Register the logged model in MLflow Model Registry
    mlflow.register_model(
        model_uri=f"runs:/{run.info.run_id}/{MODEL_NAME}", 
        name=MODEL_NAME
    )

logger.info(f'Registered the model: {MODEL_NAME}')

2025-10-14 19:37:20 - INFO - Starting the experiment: Two_Cities_Experiment
2025-10-14 19:37:21 - INFO - Run's Artifact URI: /phoenix/mlflow/884083612565343183/19a9ab2d4a68437e928625f0c600f09c/artifacts
Registered model 'Two_Cities_Model' already exists. Creating a new version of this model...
2025/10/14 19:37:33 WARNING mlflow.tracking._model_registry.fluent: Run with id 19a9ab2d4a68437e928625f0c600f09c has no artifacts at artifact path 'Two_Cities_Model', registering model based on models:/m-7e32f5883bb340f19ffd815de23c8afc instead
Created version '2' of model 'Two_Cities_Model'.
2025-10-14 19:37:36 - INFO - Registered the model: Two_Cities_Model


## Fetching the Latest Model Version from MLflow

In [11]:
# Initialize the MLflow client
client = MlflowClient()

# Retrieve the latest version of the model
model_metadata = client.get_latest_versions(MODEL_NAME, stages=["None"])
latest_model_version = model_metadata[0].version  # Extract the latest model version

# Fetch model information, including its signature
model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_model_version}")

# Print the latest model version and its signature
print(f"Latest Model Version: {latest_model_version}")
print(f"Model Signature: {model_info.signature}")

Latest Model Version: 2
Model Signature: inputs: 
  ['city': string (required), 'steps': long (required)]
outputs: 
  ['class': string (required)]
params: 
  None



## Loading the Model and Running Inference

In [12]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_model_version}")
# Make predictions for New York
ny_prediction = model.predict({
    "city": ["New York"],
    "steps": [3]  # Forecast for the next 3 days
})

# Make predictions for London
ldn_prediction = model.predict({
    "city": ["London"],
    "steps": [3]  # Forecast for the next 3 days
})

print(f"New York: {ny_prediction}" )
print("\n")
print(f"London: {ldn_prediction}")

New York:                             retail_forecast  pharmacy_forecast  \
2025-10-14 19:37:41.665796       -33.728463         -30.846299   
2025-10-15 19:37:41.665796       -38.215550         -30.564491   
2025-10-16 19:37:41.665796       -35.413120         -22.604498   

                            parks_forecast  transit_station_forecast  \
2025-10-14 19:37:41.665796       10.141208                -22.428500   
2025-10-15 19:37:41.665796       -4.275643                -39.144546   
2025-10-16 19:37:41.665796       11.427796                -30.944809   

                            workplaces_forecast  case_count_forecast  \
2025-10-14 19:37:41.665796           -22.994693          2408.753695   
2025-10-15 19:37:41.665796           -56.921910          4896.935950   
2025-10-16 19:37:41.665796           -40.361528          4383.142165   

                            hospitalized_count_forecast  death_count_forecast  
2025-10-14 19:37:41.665796                    98.599755            

In [13]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

2025-10-14 19:37:41 - INFO - ⏱️ Total execution time: 0m 21.55s
2025-10-14 19:37:41 - INFO - ✅ Notebook execution completed successfully.


Built with ❤️ using [**HP AI Studio**](https://hp.com/ai-studio).